# RealMLP - FE

This notebook trains `RealMLP` with features from the following notebooks:

- https://www.kaggle.com/code/yekenot/ps-s6-e3-realmlp-pytabkit (@yekenot)
- target encoding logits - https://www.kaggle.com/code/cdeotte/chatgpt-vibe-coding-3xgpu-models-cv-0-9178 (@cdeotte)
- groupby/agg orig - lots of sources, I've used my code : https://www.kaggle.com/code/include4eto/single-xgb-cudf-pseudo-labels-cv-0-91789
- distance to orig from - https://www.kaggle.com/code/blamerx/s6e3-0-91925-bi-tri-gram-te-xgb (@blamerx)
  

In [1]:
!pip install -q pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 6.3 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
import keras

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

from itertools import combinations
from pytabkit import RealMLP_TD_Classifier

2026-03-10 22:28:24.919770: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773181705.139828      26 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773181705.204020      26 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773181705.723316      26 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773181705.723362      26 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773181705.723365      26 computation_placer.cc:177] computation placer alr

In [3]:
import os
import random
import warnings
import numpy as np, pandas as pd
from colorama import Fore, Style
from importlib.metadata import version
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

In [4]:
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    keras.utils.set_random_seed(812)
    
seed_everything(seed=42)

# Load Data

In [5]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")
orig = pd.read_csv('/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv')

_tc = pd.to_numeric(orig.TotalCharges, errors='coerce')
_tc[_tc.isnull()] = orig[_tc.isnull()].MonthlyCharges * orig[_tc.isnull()].tenure
orig.TotalCharges = _tc
print("Train shape:", train.shape)
print("Test shape :", test.shape)

Train shape: (594194, 21)
Test shape : (254655, 20)


# Preprocessing/FE

In [6]:
def pctrank_against(values, reference):
    ref_sorted = np.sort(reference)
    return (np.searchsorted(ref_sorted, values) / len(ref_sorted)).astype('float32')

def zscore_against(values, reference):
    mu, sigma = np.mean(reference), np.std(reference)
    return (np.zeros(len(values), dtype='float32') if sigma == 0 
            else ((values - mu) / sigma).astype('float32'))

In [7]:
%%time
ID = 'id'
TARGET = 'Churn'
train[TARGET] = train[TARGET].map({'Yes': 1, 'No': 0})
orig[TARGET] = orig[TARGET].map({'Yes': 1, 'No': 0})
X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET]
X_test = test.drop([ID], axis=1); test_id = test[ID]
X_orig = orig.drop(['customerID', TARGET], axis=1);
y_orig = orig[TARGET]

print("X      init shape:", X.shape)
print("X_test init shape:", X_test.shape, "\n")

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
_non_eng_cat_cols = list(cat_cols)
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
def feature_engineering(df, fit=False):
    SERVICE_COLS = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

    df['_service_count'] = (df[SERVICE_COLS] == 'Yes').sum(axis=1).astype('float32')
    df['_has_internet'] = (df['InternetService'] != 'No').astype('float32')
    df['_has_phone'] = (df['PhoneService'] == 'Yes').astype('float32')


    for col in num_cols:
        freq = df.value_counts(normalize=True)
        df[f'_FREQ_{col}'] = df[col].map(freq).fillna(0).astype('float32')
    
    
    # Arithmetic interaction
    df['_charges_deviation'] = (df['TotalCharges'] - df['tenure'] * df['MonthlyCharges']).astype('float32')
    
    df['_MonthlyCharges_DIV_TotalCharges'] = (df['MonthlyCharges'] / (df['TotalCharges'] + 1e-6)).astype('float32')
    df['_TotalCharges_DIV_tenure']  = (df['TotalCharges'] / (df['tenure'] + 1e-6)).astype('float32')
    df['_TotalCharges_DIV_MonthlyCharges']  = (df['TotalCharges'] / (df['MonthlyCharges'] + 1e-6)).astype('float32')
    df['_Monthly_to_avg_ratio'] = (df['MonthlyCharges'] / (df['_TotalCharges_DIV_tenure'] + 1e-6)).astype('float32')
    df['_tenure_sq'] = (df['tenure'] ** 2).astype("float32")
    
    # Digit extraction
    for col in ['TotalCharges']:
        k = -3
        digit_name = f'{col}_d{k}_'
        df[digit_name] = ((df[col] * 10**k) % 10).astype('int8')
        df[digit_name] = df[digit_name].astype('category')

    # Discretize numericals
    bin_config = {'MonthlyCharges': 200, 'TotalCharges': 4000, 'tenure': 25}
    for col in ['MonthlyCharges', 'TotalCharges', 'tenure']:
        n_bins = bin_config[col]
        bin_name = f"{col}_bin_"
        if fit:
            kb = KBinsDiscretizer(
                n_bins=n_bins,
                encode='ordinal',
                strategy='quantile',
                subsample=None,
            )
            binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
            category_map[bin_name] = kb
        else:
            kb = category_map[bin_name]
            binned = kb.transform(df[[col]]).ravel().astype('int32')
        df[bin_name] = binned
        df[bin_name] = df[bin_name].astype('category')

    # ORIG probabilities
    for col in cat_cols + num_cols:
        tmp = orig.groupby(col)[TARGET].mean()
        _name = f"_ORIG_proba_{col}"
        df = df.merge(tmp.rename(_name), on=col, how="left")
        df[_name] = df[_name].fillna(0.5).astype('float32')

    # PCT ranked features
    # ===============================================================
    orig_churner_tc    = orig.loc[orig[TARGET] == 1, 'TotalCharges'].values
    orig_nonchurner_tc = orig.loc[orig[TARGET] == 0, 'TotalCharges'].values
    orig_tc            = orig['TotalCharges'].values
    orig_is_mc_mean    = orig.groupby('InternetService')['MonthlyCharges'].mean()


    tc = df['TotalCharges'].values
    df['_pctrank_nonchurner_TC']  = pctrank_against(tc, orig_nonchurner_tc)
    df['_pctrank_churner_TC']     = pctrank_against(tc, orig_churner_tc)
    df['_pctrank_orig_TC']        = pctrank_against(tc, orig_tc)
    df['_zscore_churn_gap_TC'] = (np.abs(zscore_against(tc, orig_churner_tc)) - 
                                 np.abs(zscore_against(tc, orig_nonchurner_tc))).astype('float32')
    df['_zscore_nonchurner_TC'] = zscore_against(tc, orig_nonchurner_tc)
    df['_pctrank_churn_gap_TC'] = (pctrank_against(tc, orig_churner_tc) - 
                                  pctrank_against(tc, orig_nonchurner_tc)).astype('float32')
    df['_resid_IS_MC'] = (df['MonthlyCharges'] - df['InternetService'].map(orig_is_mc_mean).fillna(0)).astype('float32')
    
    vals = np.zeros(len(df), dtype='float32')
    for cat_val in orig['InternetService'].unique():
        mask = df['InternetService'] == cat_val
        ref = orig.loc[orig['InternetService'] == cat_val, 'TotalCharges'].values
        if len(ref) > 0 and mask.sum() > 0:
            vals[mask] = pctrank_against(df.loc[mask, 'TotalCharges'].values, ref)
    df['_cond_pctrank_IS_TC'] = vals
    
    vals = np.zeros(len(df), dtype='float32')
    for cat_val in orig['Contract'].unique():
        mask = df['Contract'] == cat_val
        ref = orig.loc[orig['Contract'] == cat_val, 'TotalCharges'].values
        if len(ref) > 0 and mask.sum() > 0:
            vals[mask] = pctrank_against(df.loc[mask, 'TotalCharges'].values, ref)
    df['_cond_pctrank_C_TC'] = vals

    # ===============================================================

    for q_label, q_val in [('q25', 0.25), ('q50', 0.50), ('q75', 0.75)]:
        ch_q = np.quantile(orig_churner_tc, q_val)
        nc_q = np.quantile(orig_nonchurner_tc, q_val)
            
        df[f'_dist_To_ch_{q_label}'] = np.abs(df['TotalCharges'] - ch_q).astype('float32')
        df[f'_dist_To_nc_{q_label}'] = np.abs(df['TotalCharges'] - nc_q).astype('float32')
        df[f'_qdist_gap_To_{q_label}'] = (df[f'_dist_To_nc_{q_label}'] - df[f'_dist_To_ch_{q_label}']).astype('float32')

    # ===============================================================
                
    # Categorize numericals
    for col in num_cols:
        cat_name = f"{col}_cat_"
        round_level = 0
        if fit:
            round_flag = col == 'TotalCharges'
            series = df[col].round(round_level) if round_flag else df[col]
            codes, uniques = series.factorize()
            category_map[col] = {'uniques': uniques, 'round_flag': round_flag}
        else:
            round_flag = category_map[col]['round_flag']
            uniques = category_map[col]['uniques']
            series = df[col].round(round_level) if round_flag else df[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = series.map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes    
        df[cat_name] = df[cat_name].astype('category')

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols

df_all = pd.concat([X, X_test, X_orig], axis=0).reset_index(drop=True)
print(len(df_all))
df_tf, new_cat_cols, new_num_cols = feature_engineering(df_all, fit=True)

X = df_tf.iloc[:len(train)]
X_test = df_tf.iloc[len(train):len(train) + len(test)]
X_orig = df_tf.iloc[len(train) + len(test):]

cat_cols += new_cat_cols; num_cols += new_num_cols
print("len(new_cat_cols):", len(new_cat_cols))
print("len(new_num_cols):", len(new_num_cols), "\n")

print("prep len(cat_cols):", len(cat_cols))
print("prep len(num_cols):", len(num_cols), "\n")
print("X      prep shape:", X.shape)
print("X_test prep shape:", X_test.shape, "\n")

X      init shape: (594194, 19)
X_test init shape: (254655, 19) 

init len(cat_cols): 15
init len(num_cols): 4 

855892
len(new_cat_cols): 8
len(new_num_cols): 50 

prep len(cat_cols): 23
prep len(num_cols): 54 

X      prep shape: (594194, 77)
X_test prep shape: (254655, 77) 

CPU times: user 19.6 s, sys: 5.64 s, total: 25.2 s
Wall time: 25.2 s


In [8]:
for c in cat_cols:
    X[c] = X[c].astype(str).astype('category')
    X_test[c] = X_test[c].astype(str).astype('category')
    X_orig[c] = X_orig[c].astype(str).astype('category')

# Model Training

In [9]:
realmlp_params = {
    'device': 'cuda',
    'random_state': 42,
    'verbosity': 2,
    'n_epochs': 10,
    'batch_size': 256,
    'n_ens': 8,
    'val_metric_name': '1-auc_ovr',
    'use_early_stopping': True,
    'early_stopping_additive_patience': 20,
    'early_stopping_multiplicative_patience': 1,
    'act': "mish",
    'embedding_size': 6,
    'first_layer_lr_factor': 0.25,
    'hidden_sizes': "rectangular",
    'hidden_width': 352,
    'lr': 0.075,
    'ls_eps': 0.01,
    'ls_eps_sched': "coslog4",
    'max_one_hot_cat_size': 18,
    'n_hidden_layers': 4,
    'p_drop': 0.05,
    'p_drop_sched': "flat_cos",
    'plr_hidden_1': 16,
    'plr_hidden_2': 8,
    'plr_lr_factor': 0.1151,
    'plr_sigma': 2.33,
    'scale_lr_factor': 2.24,
    'sq_mom': 0.988,
    'wd': 0.0236,
}

In [10]:
import gc

In [11]:
# ============================================================
# CONFIG
# ============================================================
SEED = 0
N_SPLITS = 5
TE_INNER_SPLITS = 5

USE_LOGIT3 = True
TE_SMOOTH = 20.0
MAX_PAIRS = None

outer_kfold = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# ============================================================
# BASIC COLUMN SETUP
# ============================================================
X = X.copy()
X_test = X_test.copy()
X_orig = X_orig.copy()

cat_cols = X.select_dtypes(include=['object', 'category', 'string', 'bool']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

for c in cat_cols:
    X[c] = X[c].astype("string").fillna("__MISSING__")
    X_test[c] = X_test[c].astype("string").fillna("__MISSING__")
    X_orig[c] = X_orig[c].astype("string").fillna("__MISSING__")

for c in num_cols:
    X[c] = X[c].astype("float32")
    X_test[c] = X_test[c].astype("float32")
    X_orig[c] = X_orig[c].astype("float32")

# pair list: categorical pairs only
pair_cols = list(combinations(_non_eng_cat_cols, 2))
if MAX_PAIRS is not None:
    pair_cols = pair_cols[:MAX_PAIRS]

print(f"num cat cols: {len(cat_cols)}")
print(f"num num cols: {len(num_cols)}")
print(f"num TE pairs: {len(pair_cols)}")

# ============================================================
# TE HELPERS
# ============================================================
def make_pair_key(df, f1, f2):
    return df[f1].astype(str) + "___" + df[f2].astype(str)

def add_smoothed_mean_encoding(train_keys, train_y, apply_keys, smooth=20.0):
    global_mean = float(np.mean(train_y))

    stats = pd.DataFrame({
        "key": train_keys,
        "y": np.asarray(train_y, dtype=np.float32),
    }).groupby("key")["y"].agg(["mean", "count"])

    enc = (stats["mean"] * stats["count"] + global_mean * smooth) / (stats["count"] + smooth)
    return apply_keys.map(enc).fillna(global_mean).astype(np.float32).values

def build_pair_te_features(
    Xtr, ytr, Xva, Xte, pair_cols,
    inner_splits=5, seed=0, smooth=20.0, use_logit3=True
):
    """
    Leak-free TE:
      - train rows get inner-fold OOF TE
      - valid/test rows get TE fitted on all Xtr/ytr
    """
    n_tr = len(Xtr)
    n_va = len(Xva)
    n_te = len(Xte)
    n_pair = len(pair_cols)

    tr_pair = np.zeros((n_tr, n_pair), dtype=np.float32)
    va_pair = np.zeros((n_va, n_pair), dtype=np.float32)
    te_pair = np.zeros((n_te, n_pair), dtype=np.float32)

    inner_kf = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=seed)

    ytr_np = np.asarray(ytr)

    for j, (f1, f2) in enumerate(pair_cols):
        if j == 0 or (j + 1) % 20 == 0 or (j + 1) == n_pair:
            print(f"  pair {j+1:>4d}/{n_pair}: {f1} x {f2}")

        tr_keys_full = make_pair_key(Xtr, f1, f2)
        va_keys = make_pair_key(Xva, f1, f2)
        te_keys = make_pair_key(Xte, f1, f2)

        # OOF encodings for training rows
        tr_oof = np.zeros(n_tr, dtype=np.float32)
        for it, iv in inner_kf.split(Xtr, ytr_np):
            fit_keys = tr_keys_full.iloc[it]
            fit_y = ytr_np[it]
            val_keys = tr_keys_full.iloc[iv]

            tr_oof[iv] = add_smoothed_mean_encoding(
                fit_keys, fit_y, val_keys, smooth=smooth
            )

        tr_pair[:, j] = tr_oof

        # full-train encoding for valid/test
        va_pair[:, j] = add_smoothed_mean_encoding(
            tr_keys_full, ytr_np, va_keys, smooth=smooth
        )
        te_pair[:, j] = add_smoothed_mean_encoding(
            tr_keys_full, ytr_np, te_keys, smooth=smooth
        )

    if use_logit3:
        eps = 1e-5

        def logit3(m):
            m = np.clip(m, eps, 1.0 - eps)
            z = np.log(m / (1.0 - m)).astype(np.float32)
            return np.hstack([z, z**2, z**3]).astype(np.float32)

        tr_pair = logit3(tr_pair)
        va_pair = logit3(va_pair)
        te_pair = logit3(te_pair)

    return tr_pair, va_pair, te_pair

def make_te_df(arr, base_names):
    return pd.DataFrame(arr, columns=base_names)

# ============================================================
# BUILD TE COLUMN NAMES
# ============================================================
base_pair_names = [f"te__{f1}__{f2}" for (f1, f2) in pair_cols]
if USE_LOGIT3:
    te_col_names = (
        [f"{c}__z1" for c in base_pair_names] +
        [f"{c}__z2" for c in base_pair_names] +
        [f"{c}__z3" for c in base_pair_names]
    )
else:
    te_col_names = base_pair_names

# ============================================================
# TRAIN
# ============================================================
oof = np.zeros(len(X), dtype=np.float32)
test = np.zeros(len(X_test), dtype=np.float32)

for fold, (t, v) in enumerate(outer_kfold.split(X, y), 1):
    print("\n" + "=" * 80)
    print(f"fold {fold}")
    print("=" * 80)

    Xt = X.iloc[t].copy().reset_index(drop=True)
    Xv = X.iloc[v].copy().reset_index(drop=True)
    yt = y.iloc[t].copy()
    yv = y.iloc[v].copy()

    # This can't be used as-is with our feature engineering because orig['Churn'] is part of the pipeline
    # if USE_X_ORIG:
    #     Xt = pd.concat([Xt, X_orig], axis=0).reset_index(drop=True)
    #     yt = pd.Series(np.concatenate([yt.values, np.asarray(y_orig)]), name=y.name)
    # else:
    yt = yt.reset_index(drop=True)

    X_test_fold = X_test.copy().reset_index(drop=True)

    # Pairwise TE features
    tr_te, va_te, te_te = build_pair_te_features(
        Xt, yt.values, Xv, X_test_fold,
        pair_cols=pair_cols,
        inner_splits=TE_INNER_SPLITS,
        seed=SEED + fold,
        smooth=TE_SMOOTH,
        use_logit3=USE_LOGIT3
    )

    # Optional: scale TE features only
    # RealMLP already preprocesses numeric features internally,
    # but keeping this often helps when TE/logit3 features get wide/heavy-tailed.
    scaler_te = StandardScaler()
    tr_te = scaler_te.fit_transform(tr_te).astype(np.float32)
    va_te = scaler_te.transform(va_te).astype(np.float32)
    te_te = scaler_te.transform(te_te).astype(np.float32)

    tr_te_df = make_te_df(tr_te, te_col_names)
    va_te_df = make_te_df(va_te, te_col_names)
    te_te_df = make_te_df(te_te, te_col_names)

    # Final dataframes for RealMLP
    Xt_model = pd.concat([Xt.reset_index(drop=True), tr_te_df], axis=1)
    Xv_model = pd.concat([Xv.reset_index(drop=True), va_te_df], axis=1)
    Xte_model = pd.concat([X_test_fold.reset_index(drop=True), te_te_df], axis=1)

    # ensure categorical columns stay categorical/string
    for c in cat_cols:
        Xt_model[c] = Xt_model[c].astype("string")
        Xv_model[c] = Xv_model[c].astype("string")
        Xte_model[c] = Xte_model[c].astype("string")

    # ensure all TE cols are float32
    for c in te_col_names:
        Xt_model[c] = Xt_model[c].astype("float32")
        Xv_model[c] = Xv_model[c].astype("float32")
        Xte_model[c] = Xte_model[c].astype("float32")

    model = RealMLP_TD_Classifier(**realmlp_params)

    model.fit(
        Xt_model,
        yt,
        Xv_model,
        yv,
        cat_col_names=cat_cols,   # explicit is safer
    )

    val_preds = model.predict_proba(Xv_model)[:, 1]
    fold_test_preds = model.predict_proba(Xte_model)[:, 1]

    oof[v] = val_preds
    test += fold_test_preds / N_SPLITS

    fold_score = roc_auc_score(yv, val_preds)
    print(f"fold {fold} auc: {fold_score:.6f}")

    del model, Xt_model, Xv_model, Xte_model, tr_te, va_te, te_te
    gc.collect()
    torch.cuda.empty_cache()
    
print("OOF score: %.5f" % roc_auc_score(y, oof))

num cat cols: 23
num num cols: 54
num TE pairs: 105

fold 1
  pair    1/105: gender x Partner
  pair   20/105: Partner x OnlineBackup
  pair   40/105: PhoneService x MultipleLines
  pair   60/105: MultipleLines x PaymentMethod
  pair   80/105: OnlineBackup x StreamingTV
  pair  100/105: StreamingMovies x Contract
  pair  105/105: PaperlessBilling x PaymentMethod
Columns classified as continuous: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', '_service_count', '_has_internet', '_has_phone', '_FREQ_SeniorCitizen', '_FREQ_tenure', '_FREQ_MonthlyCharges', '_FREQ_TotalCharges', '_charges_deviation', '_MonthlyCharges_DIV_TotalCharges', '_TotalCharges_DIV_tenure', '_TotalCharges_DIV_MonthlyCharges', '_Monthly_to_avg_ratio', '_tenure_sq', '_ORIG_proba_gender', '_ORIG_proba_Partner', '_ORIG_proba_Dependents', '_ORIG_proba_PhoneService', '_ORIG_proba_MultipleLines', '_ORIG_proba_InternetService', '_ORIG_proba_OnlineSecurity', '_ORIG_proba_OnlineBackup', '_ORIG_proba_DeviceProtecti

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/10: val 1-auc_ovr = 0.081840
Epoch 2/10: val 1-auc_ovr = 0.081136
Epoch 3/10: val 1-auc_ovr = 0.081865
Epoch 4/10: val 1-auc_ovr = 0.081203
Epoch 5/10: val 1-auc_ovr = 0.081072
Epoch 6/10: val 1-auc_ovr = 0.081408
Epoch 7/10: val 1-auc_ovr = 0.081554
Epoch 8/10: val 1-auc_ovr = 0.081094
Epoch 9/10: val 1-auc_ovr = 0.080836
Epoch 10/10: val 1-auc_ovr = 0.081830


`Trainer.fit` stopped: `max_epochs=10` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


fold 1 auc: 0.919164

fold 2
  pair    1/105: gender x Partner
  pair   20/105: Partner x OnlineBackup
  pair   40/105: PhoneService x MultipleLines
  pair   60/105: MultipleLines x PaymentMethod
  pair   80/105: OnlineBackup x StreamingTV
  pair  100/105: StreamingMovies x Contract
  pair  105/105: PaperlessBilling x PaymentMethod
Columns classified as continuous: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', '_service_count', '_has_internet', '_has_phone', '_FREQ_SeniorCitizen', '_FREQ_tenure', '_FREQ_MonthlyCharges', '_FREQ_TotalCharges', '_charges_deviation', '_MonthlyCharges_DIV_TotalCharges', '_TotalCharges_DIV_tenure', '_TotalCharges_DIV_MonthlyCharges', '_Monthly_to_avg_ratio', '_tenure_sq', '_ORIG_proba_gender', '_ORIG_proba_Partner', '_ORIG_proba_Dependents', '_ORIG_proba_PhoneService', '_ORIG_proba_MultipleLines', '_ORIG_proba_InternetService', '_ORIG_proba_OnlineSecurity', '_ORIG_proba_OnlineBackup', '_ORIG_proba_DeviceProtection', '_ORIG_proba_TechSupport',

...

fold 4 auc: 0.918937

fold 5
  pair    1/105: gender x Partner
  pair   20/105: Partner x OnlineBackup
  pair   40/105: PhoneService x MultipleLines
  pair   60/105: MultipleLines x PaymentMethod
  pair   80/105: OnlineBackup x StreamingTV
  pair  100/105: StreamingMovies x Contract
  pair  105/105: PaperlessBilling x PaymentMethod
Columns classified as continuous: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', '_service_count', '_has_internet', '_has_phone', '_FREQ_SeniorCitizen', '_FREQ_tenure', '_FREQ_MonthlyCharges', '_FREQ_TotalCharges', '_charges_deviation', '_MonthlyCharges_DIV_TotalCharges', '_TotalCharges_DIV_tenure', '_TotalCharges_DIV_MonthlyCharges', '_Monthly_to_avg_ratio', '_tenure_sq', '_ORIG_proba_gender', '_ORIG_proba_Partner', '_ORIG_proba_Dependents', '_ORIG_proba_PhoneService', '_ORIG_proba_MultipleLines', '_ORIG_proba_InternetService', '_ORIG_proba_OnlineSecurity', '_ORIG_proba_OnlineBackup', '_ORIG_proba_DeviceProtection', '_ORIG_proba_TechSupport',

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/10: val 1-auc_ovr = 0.082601
Epoch 2/10: val 1-auc_ovr = 0.081785
Epoch 3/10: val 1-auc_ovr = 0.082304
Epoch 4/10: val 1-auc_ovr = 0.081804
Epoch 5/10: val 1-auc_ovr = 0.081943
Epoch 6/10: val 1-auc_ovr = 0.081974
Epoch 7/10: val 1-auc_ovr = 0.082179
Epoch 8/10: val 1-auc_ovr = 0.081649
Epoch 9/10: val 1-auc_ovr = 0.081633
Epoch 10/10: val 1-auc_ovr = 0.082903


`Trainer.fit` stopped: `max_epochs=10` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


fold 5 auc: 0.918367
OOF score: 0.91843


# OOF and Submission

In [12]:
print('OOF score: %.5f' % roc_auc_score(y, oof))

OOF score: 0.91843


In [13]:
df_oof = pd.DataFrame({'Churn': oof})
df_oof.to_csv(f'oof.csv', index=False)

df_test = pd.DataFrame({'Churn': test})
df_test.to_csv(f'test.csv',index=False)

print("Saved oof to file")

Saved oof to file


In [14]:
sub = pd.DataFrame({
    'id': test_id,
    'Churn': test
})

sub.to_csv(f'submission.csv', index=False)